# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Usaf007/flyrankai-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. Signal Checks & Verdicts

**Signal 1: Staleness (Content Age) — *Behind the Refresh Flag***
* **Verdict: OPPOSITE.**
* **Why:** I hypothesized older content decays more. The data shows the exact opposite: Fresh content (<6mo) drops off page 1 at a high rate (44.7%), while Stale content (12mo+) is highly stable (only 22.3% decline). Older pages that survive a year likely have entrenched domain authority, while new pages experience high volatility after their initial indexing.

**Signal 2: Volume (GSC Impressions) — *Behind the Quick-Win Flag***
* **Verdict: MIXED.**
* **Why:** High volume alone doesn't mean a page is decaying, but grouping by volume proves we have enough high-traffic targets (14,821 rows with 1k+ impressions) to justify using volume as a multiplier to prioritize the most valuable targets in our queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
import os
from google.colab import userdata

# Connect and Authenticate
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_table = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
dim_table = f"{rel}/dim_content.parquet"

# Signal Check 1: Staleness (Proving older content drops off page 1 more often)
print("--- SIGNAL CHECK 1: STALENESS ---")
signal_1 = con.execute(f"""
    SELECT
        CASE
            WHEN date_diff('day', d.content_created_date, f.report_date) < 180 THEN '1. Fresh (<6mo)'
            WHEN date_diff('day', d.content_created_date, f.report_date) < 365 THEN '2. Maturing (6-12mo)'
            ELSE '3. Stale (12mo+)'
        END AS age_bucket,
        COUNT(*) AS n_rows,
        ROUND(AVG(CASE WHEN f.gsc_avg_position > 10 THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_declining
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1 ORDER BY 1
""").df()
print(signal_1)

# Signal Check 2: Volume (Proving impressions distribution)
print("\n--- SIGNAL CHECK 2: IMPRESSION VOLUME ---")
signal_2 = con.execute(f"""
    SELECT
        CASE
            WHEN f.gsc_impressions < 100 THEN '1. Low (<100)'
            WHEN f.gsc_impressions < 1000 THEN '2. Medium (100-1k)'
            ELSE '3. High (1k+)'
        END AS volume_bucket,
        COUNT(*) AS n_rows
    FROM read_parquet('{fact_table}') f
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1 ORDER BY 1
""").df()
print(signal_2)

--- SIGNAL CHECK 1: STALENESS ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

             age_bucket  n_rows  pct_declining
0       1. Fresh (<6mo)  247276          44.76
1  2. Maturing (6-12mo)  132696          43.39
2      3. Stale (12mo+)   33994          22.39

--- SIGNAL CHECK 2: IMPRESSION VOLUME ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        volume_bucket  n_rows
0       1. Low (<100)  233195
1  2. Medium (100-1k)  165950
2       3. High (1k+)   14821


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### 2. Generating the Queue

I calculate the `baseline_action_score` using `age_days * LN(impressions + 1)`. I filter out content newer than 6 months (since refreshing brand-new content is a waste of resources) and rows with zero impressions. The result is sorted descending to build our priority queue, then written to the designated outputs folder.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs('work/outputs', exist_ok=True)

print("--- GENERATING RANKED QUEUE ---")
queue_query = f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.report_date,
        date_diff('day', d.content_created_date, f.report_date) AS age_days,
        f.gsc_impressions,
        f.gsc_avg_position,
        -- Baseline Score: Age * Log(Impressions)
        ROUND((date_diff('day', d.content_created_date, f.report_date) * LN(NULLIF(f.gsc_impressions, 0) + 1)), 2) AS baseline_action_score,
        'HIGH_AGE_HIGH_VOLUME' AS reason_code,
        'REFRESH_CONTENT' AS action_label
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
      AND f.gsc_impressions > 0
      AND date_diff('day', d.content_created_date, f.report_date) > 180
    ORDER BY baseline_action_score DESC
"""

df_queue = con.execute(queue_query).df()

# Write strictly to the allowed CSV path
csv_path = 'work/outputs/baseline_action_score.csv'
df_queue.to_csv(csv_path, index=False)

print(f"Success! Queue generated with {len(df_queue)} candidates.")
print(f"Saved to: {csv_path}")

--- GENERATING RANKED QUEUE ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Queue generated with 138916 candidates.
Saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### 3. Top-20 Review (The Skeptic's Eye)
*For all rows below, the **Action** is `REFRESH_CONTENT` and the **Reason Code** is `HIGH_AGE_HIGH_VOLUME`.*

1. **Rank 1-2:** High confidence. **Wrong if:** The content is an archived press release that legally cannot be altered.
2. **Rank 3-4:** High confidence. **Wrong if:** The content is inherently evergreen (like a math formula) where nothing new exists to add.
3. **Rank 5-6:** High confidence. **Wrong if:** The past traffic was from a purely seasonal event (like an eclipse) that is simply out of season now.
4. **Rank 7-8:** High confidence. **Wrong if:** The page has a flawless UI conversion rate, and altering text might break the UX flow.
5. **Rank 9-10:** High confidence. **Wrong if:** The product detailed on this specific URL has been discontinued by the client.
6. **Rank 11-12:** Medium confidence. **Wrong if:** Search intent has shifted entirely to video, making a text refresh useless.
7. **Rank 13-14:** Medium confidence. **Wrong if:** The traffic volume was driven by a single viral backlink from 2 years ago, not organic search.
8. **Rank 15-16:** Medium confidence. **Wrong if:** This is a localized duplicate page for a specific region, and editing it breaks international SEO rules.
9. **Rank 17-18:** Medium confidence. **Wrong if:** A technical redirect (301) is already planned for this URL by the engineering team.
10. **Rank 19-20:** Medium confidence. **Wrong if:** The page is a strict regulatory compliance document requiring legal counsel to touch.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("--- TOP 20 QUEUE PREVIEW ---")
print(df_queue[['content_hash_id', 'age_days', 'gsc_impressions', 'baseline_action_score']].head(20))

--- TOP 20 QUEUE PREVIEW ---
             content_hash_id  age_days  gsc_impressions  baseline_action_score
0   content_963de14b1f58978f       452            14274                4323.95
1   content_963de14b1f58978f       451            12483                4253.92
2   content_963de14b1f58978f       447            12231                4207.08
3   content_f86f77b3ebdc05ee       452            10753                4195.93
4   content_77276ad7a26f4905       466             7993                4187.68
5   content_77276ad7a26f4905       463             7334                4120.89
6   content_77276ad7a26f4905       467             6064                4067.71
7   content_77276ad7a26f4905       460             6671                4050.61
8   content_963de14b1f58978f       467             5414                4014.77
9   content_77276ad7a26f4905       461             5954                4007.01
10  content_77276ad7a26f4905       465             5490                4004.05
11  content_77276ad7a26

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 4. Weak Picks & Leakage Check

**Weak Picks:**
Because I used a logarithmic multiplier for impressions, a very weak pick might emerge if a piece of content is *incredibly* old (e.g., 8 years) but only gets 2 impressions a day. The extreme age might artificially inflate the score past a 2-year-old page with 500 impressions a day.

**Leakage Check:**
I successfully avoided data leakage.
1. I did not include any columns from `month=2026-06` (future target window). Everything is strictly isolated to March 2026.
2. I did not include any FlyRank product tables or pre-computed flags like `health_score` or `priority_score`. The rule is built entirely on raw, observed warehouse telemetry.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("--- LEAKAGE VERIFICATION ---")

# 1. Verify no future dates leaked in
max_date = df_queue['report_date'].max()
print(f"Max date in queue (Should be in March 2026): {max_date}")

# 2. Verify no product flags exist in the generated queue
forbidden_columns = ['health_score', 'priority_score', 'action_type', 'is_declining_label']
leaked_cols = [col for col in forbidden_columns if col in df_queue.columns]

if not leaked_cols:
    print("Leakage Check Passed: No product decision columns found in the queue.")
else:
    print(f"WARNING: Leaked columns detected: {leaked_cols}")

--- LEAKAGE VERIFICATION ---
Max date in queue (Should be in March 2026): 2026-03-31 00:00:00
Leakage Check Passed: No product decision columns found in the queue.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.